# Calculator And Converter Agent

In this notebook, I build a small agent that can do arithmetic and convert between everyday units, using the Hugging Face `smolagents` library.

Unlike the retrieval agent in the previous notebook, this one does not look anything up. It solves the question itself by calling tools that do exact, reliable calculations, instead of asking the language model to do arithmetic in its head, which is a common source of mistakes.

In this notebook, I will learn how to:

- Write a plain Python calculator function and a plain Python unit converter function
- Turn each one into an agent tool with the `@tool` decorator
- Handle bad input, like dividing by zero, inside a tool
- Give an agent more than one tool and watch it choose the right one
- See what this kind of simple agent still cannot do

Everything here stays small on purpose, so the whole idea fits in one sitting.

## 1. Importing Libraries and Creating the Model

First I import the pieces I need from `smolagents`.

`CodeAgent` is the agent that writes and runs Python code to solve a task. `tool` is the decorator I use to turn a plain function into something an agent can call. `InferenceClientModel` is the language model, which runs on the Hugging Face Inference API rather than on my own machine.

In [ ]:
from smolagents import CodeAgent, InferenceClientModel, tool

model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct"
)

## 2. Writing a Basic Calculator Function

Before I build a tool, I write the calculation as an ordinary Python function.

It takes two numbers and an operation as a plain string, like `"add"` or `"multiply"`, and returns the result. Keeping the logic in a plain function first means I can test the maths on its own, without an agent or a language model anywhere near it.

In [ ]:
def calculate(a: float, b: float, operation: str) -> float:
    """Performs one arithmetic operation on two numbers."""
    if operation == "add":
        return a + b
    if operation == "subtract":
        return a - b
    if operation == "multiply":
        return a * b
    if operation == "divide":
        return a / b
    raise ValueError(f"Unknown operation: {operation}")

## 3. Testing the Calculator Function

I try the function on a few simple sums before trusting it with anything else. This is the easiest place to catch a mistake, because there is no agent involved yet, just plain Python I can read line by line.

In [ ]:
print(calculate(4, 5, "add"))
print(calculate(10, 3, "subtract"))
print(calculate(6, 7, "multiply"))
print(calculate(20, 4, "divide"))

## 4. Handling Division by Zero and Unknown Operations

Two things can go wrong with this function: dividing by zero, and passing an operation I did not plan for. Right now, dividing by zero crashes with a `ZeroDivisionError`, which is not a message I want an agent to see.

I want both failures to raise a clear, readable `ValueError` instead, because this message is exactly what the tool will hand back to the agent later. A confusing error here becomes a confusing tool result there.

In [ ]:
def calculate(a: float, b: float, operation: str) -> float:
    """Performs one arithmetic operation on two numbers."""
    if operation == "add":
        return a + b
    if operation == "subtract":
        return a - b
    if operation == "multiply":
        return a * b
    if operation == "divide":
        if b == 0:
            raise ValueError("Cannot divide by zero.")
        return a / b
    raise ValueError(
        f"Unknown operation: {operation}. Use add, subtract, multiply or divide."
    )

## 5. Testing the Error Handling

I check both failure cases directly, catching the error myself so the notebook keeps running and I can read the message that would reach the agent.

In [ ]:
for a, b, operation in [(5, 0, "divide"), (5, 3, "modulo")]:
    try:
        calculate(a, b, operation)
    except ValueError as error:
        print(error)

## 6. Turning the Calculator Into a Tool

The function works, but an agent cannot call a plain Python function. It needs a tool.

The `@tool` decorator does the conversion. `smolagents` reads the docstring to build the description the model sees, so I describe the operation argument carefully, including the exact words it should use.

In [ ]:
@tool
def calculator_tool(a: float, b: float, operation: str) -> str:
    """
    Performs one arithmetic operation on two numbers.

    Args:
        a (float): The first number.
        b (float): The second number.
        operation (str): One of "add", "subtract", "multiply" or "divide".
    """
    try:
        result = calculate(a, b, operation)
    except ValueError as error:
        return str(error)
    return str(result)